# 02 - Document Loading and Parsing

The first step in any RAG pipeline is **loading your data**. Documents come in many formats — PDFs, Markdown, web pages, CSVs, databases — and each requires a different parser.

LangChain standardizes all of these into a single `Document` object:

```python
Document(
    page_content="The actual text content...",
    metadata={"source": "file.pdf", "page": 0, ...}
)
```

This uniform representation means the rest of the pipeline (chunking, embedding, retrieval) doesn't need to know where the data came from.

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "sample")

## 1. Loading Markdown Files

Markdown is one of the simplest formats. Our loader extracts the text and preserves the document structure.

In [2]:
from rag_engine.loaders import load_documents

md_docs = load_documents(os.path.join(DATA_DIR, "rag_survey.md"))

print(f"Loaded {len(md_docs)} document(s) from Markdown")
print(f"\nMetadata: {md_docs[0].metadata}")
print(f"\nFirst 500 chars:\n{md_docs[0].page_content[:500]}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 document(s) from Markdown

Metadata: {'source': 'C:\\Users\\mathi\\Documents\\Github repos\\RAG-knowledge-engine\\data\\sample\\rag_survey.md', 'file_type': 'markdown'}

First 500 chars:
# Retrieval-Augmented Generation: A Survey

## Introduction

Retrieval-Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge at inference time. Instead of relying solely on the knowledge encoded in model weights during training, RAG systems retrieve pertinent documents from a knowledge base and include them as context in the prompt.

This approach addresses several fundamental limitations of LLMs:

- **Knowledge cu


## 2. Loading CSV Files

CSV files are loaded row by row — each row becomes a separate `Document`. Column values are combined into the `page_content`, and all columns are also stored as metadata.

In [3]:
csv_docs = load_documents(os.path.join(DATA_DIR, "ml_glossary.csv"))

print(f"Loaded {len(csv_docs)} document(s) from CSV")
print("\nFirst document:")
print(f"  Content: {csv_docs[0].page_content}")
print(f"  Metadata: {csv_docs[0].metadata}")

print("\nSecond document:")
print(f"  Content: {csv_docs[1].page_content}")
print(f"  Metadata: {csv_docs[1].metadata}")

Loaded 21 document(s) from CSV

First document:
  Content: term: Attention
definition: A mechanism that allows models to focus on relevant parts of the input when producing output. Computes weighted sums of value vectors based on query-key compatibility.
category: Architecture
  Metadata: {'source': 'C:\\Users\\mathi\\Documents\\Github repos\\RAG-knowledge-engine\\data\\sample\\ml_glossary.csv', 'row': 0, 'file_type': 'csv'}

Second document:
  Content: term: Transformer
definition: A neural network architecture based entirely on self-attention mechanisms without recurrence or convolution. Introduced in the 'Attention Is All You Need' paper.
category: Architecture
  Metadata: {'source': 'C:\\Users\\mathi\\Documents\\Github repos\\RAG-knowledge-engine\\data\\sample\\ml_glossary.csv', 'row': 1, 'file_type': 'csv'}


## 3. Loading Web Pages

The web loader fetches a URL, extracts text content (stripping HTML), and returns it as a Document.

In [6]:
web_docs = load_documents("https://en.wikipedia.org/wiki/Retrieval-augmented_generation")

print(f"Loaded {len(web_docs)} document(s) from web")
print(f"\nMetadata: {web_docs[0].metadata}")
print(f"\nFirst 500 chars:\n{web_docs[0].page_content[:500]}")

Loaded 1 document(s) from web

Metadata: {'source': 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'title': 'Retrieval-augmented generation - Wikipedia', 'language': 'en', 'file_type': 'web'}

First 500 chars:




Retrieval-augmented generation - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools





 Donate Create accou


## 4. Loading PDF Files

PDFs are loaded page by page. Each page becomes a separate Document with a `page` metadata field.

> **Note:** If you have a PDF in `data/sample/`, uncomment the cell below. PDF loading requires a valid PDF file.

In [ ]:
# Uncomment if you have a PDF in data/sample/
# pdf_docs = load_documents(os.path.join(DATA_DIR, "your_file.pdf"))
# print(f"Loaded {len(pdf_docs)} page(s) from PDF")
# print(f"Page 1 metadata: {pdf_docs[0].metadata}")
# print(f"Page 1 preview: {pdf_docs[0].page_content[:300]}")

## 5. The Unified `load_documents()` Interface

Our `load_documents()` function auto-detects the source type from the file extension or URL prefix. You can also specify it explicitly:

```python
# Auto-detect
docs = load_documents("report.pdf")            # → pdf_loader
docs = load_documents("notes.md")              # → markdown_loader
docs = load_documents("https://example.com")   # → web_loader
docs = load_documents("data.csv")              # → csv_loader

# Explicit
docs = load_documents("data.txt", source_type="markdown")
```

## Summary

| Format | Loader | Documents per file | Key metadata |
|--------|--------|-------------------|---------------|
| PDF | `PyPDFLoader` | One per page | source, page, file_type |
| Markdown | `UnstructuredMarkdownLoader` | One per file | source, file_type |
| CSV | `CSVLoader` | One per row | source, file_type, row |
| Web | `WebBaseLoader` | One per URL | source, file_type, title |

**Key takeaway:** Regardless of the source format, the output is always a list of `Document` objects with `page_content` and `metadata`. This uniformity is what makes the rest of the RAG pipeline format-agnostic.

**Next:** [03 - Chunking Strategies](./03_chunking_strategies.ipynb) — splitting these documents into retrieval-sized pieces.